# Reproduce Figures 13 and 14: DoE, synthetic-noise feature, and LightGBM bootstrap forest

This notebook reproduces Figures 13 and 14 from the Design of Experiments section of the paper using simple, readable Python. The exact DSD and I-optimal RSM layouts are loaded from the paper workbook because the original `pyDOE` package provides Latin-hypercube and classical factorial/response-surface helpers, but it does not implement JMP's Definitive Screening Design or I-optimal design generator. The notebook still uses `pyDOE.lhs` for the reproducible space-filling design path and keeps the paper workbook as the source of record for matching the published figure.

## Reference

**Main article:** Vallerio, M., del Rio Chanona, A., & Navarro-Brull, F. J. (2026). *All you need is noise - from feature selection to explainable industrial AI*. **Digital Chemical Engineering, 18**, 100290. https://doi.org/10.1016/j.dche.2026.100290

**Paper section reproduced here:** Section 4.2, *Design of Experiments - DoE*. The key idea is to compare classical DoE layouts with a space-filling Latin hypercube on a nonlinear Rosenbrock benchmark while using a Synthetic Noise Feature (SNF) as a practical cutoff for deciding whether a factor is informative.

In [ ]:
# Self-contained runtime setup for reproducing the figures.
import importlib.util
import subprocess
import sys


def ensure(import_name, package_name=None, fallback_package=None):
    """Install a package only when the current Python environment is missing it."""
    package_name = package_name or import_name
    if importlib.util.find_spec(import_name) is not None:
        return
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
    except subprocess.CalledProcessError:
        if fallback_package is None:
            raise
        subprocess.check_call([sys.executable, "-m", "pip", "install", fallback_package])


def ensure_pydoe():
    """Install pyDOE, accepting either the new `pydoe` or classic `pyDOE` import name."""
    if importlib.util.find_spec("pydoe") is not None or importlib.util.find_spec("pyDOE") is not None:
        return
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/pydoe/pydoe.git"])
    except subprocess.CalledProcessError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "pyDOE"])


for import_name, package_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("openpyxl", "openpyxl"),
    ("scipy", "scipy"),
    ("lightgbm", "lightgbm"),
    ("plotly", "plotly"),
    ("matplotlib", "matplotlib"),
    ("kaleido", "kaleido"),
    ("PIL", "pillow"),
]:
    ensure(import_name, package_name)

# Prefer the GitHub pyDOE project requested for this reproduction.
# Some older Python kernels cannot satisfy the current GitHub scipy requirement,
# so the classic PyPI pyDOE release is used as a compatible fallback.
ensure_pydoe()

# On macOS, LightGBM may also need the OpenMP runtime.
# If importing LightGBM fails with a libomp error, install it once with:
# conda install -c conda-forge llvm-openmp

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
import plotly.express as px
from scipy.interpolate import RegularGridInterpolator
from IPython.display import HTML, Image, display
from PIL import Image as PILImage
from lightgbm import LGBMRegressor
try:
    from pydoe import lhs
except ModuleNotFoundError:
    from pyDOE import lhs

warnings.filterwarnings("ignore", category=UserWarning)

DOE_ROOT = Path("/Users/b42549592/Documents/GitHub/all-you-need-is-noise/03_DoE")
DATA_PATH = DOE_ROOT / "DoEs_results_bootstrap_analysis.xlsx"
OUTPUT_DIR = DOE_ROOT / "python" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DESIGNS = ["Definitive Screening Design", "RSM (I-optimal)", "Space filling (16)"]
DISPLAY_NAMES = {
    "Definitive Screening Design": "Definitive Screening Design",
    "RSM (I-optimal)": "RSM (I-optimal)",
    "Space filling (16)": "Space filling (16)",
}
FACTOR_NAMES = ["X2", "X1", "X3 (non-informative)", "Random Normal"]
FACTOR_COLORS = {
    "X1": "#111111",
    "X2": "#111111",
    "X3 (non-informative)": "#b2182b",
    "Random Normal": "#b2182b",
}
PAPER_COLORSCALE = "Viridis"
RNG_SEED = 42


def save_plotly_png(fig, png_path, scale=2):
    """Save a static PNG from a Plotly figure for notebook-embedded paper-style display."""
    try:
        fig.write_image(png_path, scale=scale)
        print(f"Saved PNG to: {png_path}")
    except Exception as exc:
        print(f"PNG export skipped because Plotly image export is unavailable: {exc}")

## Load the paper DoE table

The workbook stores the design matrices used for the published DoE benchmark. The factor columns labelled `X1 (Rosenbrock)` and `X2 (Rosenbrock)` are the two active Rosenbrock inputs. `X3 (DoE)` is intentionally non-informative, and `Random Normal` is regenerated later as the synthetic-noise feature for the bootstrap forest.

In [ ]:
raw = pd.read_excel(DATA_PATH)

# Keep only the three designs shown in Figure 14.
doe = raw.loc[raw["Design"].isin(DESIGNS)].copy()
doe["design_label"] = pd.Categorical(doe["Design"], categories=DESIGNS, ordered=True)

# Rename the columns used in the figure to short, readable names.
doe = doe.rename(columns={
    "X1 (Rosenbrock)": "X1",
    "X2 (Rosenbrock)": "X2",
    "X3 (DoE)": "X3",
    "Y (Rosenbrock)": "Y_paper",
})

print(f"Loaded {len(doe)} runs from {DATA_PATH.name}")
display(doe.groupby("Design").size().rename("runs").to_frame())
display(doe[["Design", "X1", "X2", "X3", "Y_paper", "Random Normal"]].head())

## Rosenbrock response used in the paper

The paper uses the Rosenbrock benchmark because its optimum lies along a curved valley. The raw objective is compressed with `log10(f + 1)` and then min-max scaled so that brighter colors mean better response. This makes the nonlinear valley visible while keeping large off-valley penalties from dominating the plots.

In [ ]:
def rosenbrock_raw(x1, x2):
    """Classical two-dimensional Rosenbrock function used in the paper."""
    return (1 - x1) ** 2 + 100 * (x2 - x1 ** 2) ** 2


def scaled_rosenbrock(x1, x2, log_min=None, log_max=None):
    """Return the paper-style response: log-compressed and scaled so high is good."""
    log_y = np.log10(rosenbrock_raw(x1, x2) + 1)
    if log_min is None:
        log_min = np.nanmin(log_y)
    if log_max is None:
        log_max = np.nanmax(log_y)
    return 1 - (log_y - log_min) / (log_max - log_min)


# Use the same display domain as Figure 13.
x1_grid = np.linspace(-2, 2, 241)
x2_grid = np.linspace(-1, 3, 241)
X2_GRID, X1_GRID = np.meshgrid(x2_grid, x1_grid)
LOG_GRID = np.log10(rosenbrock_raw(X1_GRID, X2_GRID) + 1)
LOG_MIN, LOG_MAX = np.nanmin(LOG_GRID), np.nanmax(LOG_GRID)
Y_GRID = scaled_rosenbrock(X1_GRID, X2_GRID, LOG_MIN, LOG_MAX)

# Recompute the response for the loaded design points on the same scale.
doe["Y"] = scaled_rosenbrock(doe["X1"], doe["X2"], LOG_MIN, LOG_MAX)
print(doe[["Design", "X1", "X2", "Y"]].head().to_string(index=False))

## Figure 13: normalized Rosenbrock response surface

In [ ]:
figure_13_df = pd.DataFrame({
    "X1": X1_GRID.ravel(),
    "X2": X2_GRID.ravel(),
    "Y": Y_GRID.ravel(),
})

fig13 = px.density_heatmap(
    figure_13_df,
    x="X2",
    y="X1",
    z="Y",
    nbinsx=160,
    nbinsy=160,
    histfunc="avg",
    color_continuous_scale=PAPER_COLORSCALE,
    range_color=[0.40, 1.00],
    labels={"Y": "Y"},
)
fig13.update_layout(
    width=620,
    height=560,
    template="plotly_white",
    title={"text": "Normalized Rosenbrock response", "x": 0.48},
    margin={"l": 70, "r": 35, "t": 70, "b": 65},
)
fig13.update_xaxes(title="X2", range=[-1, 3], showgrid=False)
fig13.update_yaxes(title="X1", range=[-2, 2], scaleanchor="x", scaleratio=1, showgrid=False)
fig13.update_coloraxes(colorbar_title="Y")

figure_13_png_path = OUTPUT_DIR / "figure_13_normalized_rosenbrock_pydoe.png"
save_plotly_png(fig13, figure_13_png_path)
display(Image(filename=figure_13_png_path))
display(HTML(fig13.to_html(include_plotlyjs="cdn", full_html=False)))

**Figure 13 footnote.** The plot shows the two active Rosenbrock factors only. The third DoE factor is intentionally inert and is therefore not drawn. The response is log-compressed and scaled so that larger `Y` values mark better points along the curved optimum valley.

## pyDOE Latin-hypercube design helper

`pyDOE.lhs` is the part of pyDOE used here for space-filling designs. The published figure is reproduced from the workbook's exact Latin-hypercube coordinates; the helper below shows the equivalent reproducible pyDOE path for generating a fresh 3-factor Latin hypercube over the same factor bounds.

In [ ]:
def make_pydoe_lhs(samples=16, seed=RNG_SEED):
    """Create a 3-factor Latin hypercube over the paper's factor ranges."""
    np.random.seed(seed)  # pyDOE uses NumPy's global random state.
    unit = lhs(3, samples=samples, criterion="maximin", iterations=100)
    lower = np.array([-2.0, -3.0, -1.0])
    upper = np.array([2.0, 3.0, 1.0])
    scaled = lower + unit * (upper - lower)
    return pd.DataFrame(scaled, columns=["X1", "X2", "X3"])


pydoe_lh_example = make_pydoe_lhs()
print("Example pyDOE Latin-hypercube design:")
display(pydoe_lh_example.head())

## Figure 14a and 14b: design layout and response contours

Panel (a) counts repeated design coordinates after projection onto `X1` and `X2`. Panel (b) interpolates the scaled Rosenbrock response from each design's sampled points, which makes the design coverage visible: sparse corner/center-heavy designs only reconstruct a coarse valley, while the space-filling design covers the nonlinear shape more evenly.

In [ ]:
# Panel (a): projected design counts.
count_rows = []
for design in DESIGNS:
    data = doe.loc[doe["Design"] == design]
    counts = data.groupby(["X1", "X2"], as_index=False).size().rename(columns={"size": "Count"})
    counts["Design"] = DISPLAY_NAMES[design]
    count_rows.append(counts)
count_plot = pd.concat(count_rows, ignore_index=True)

fig14a = px.scatter(
    count_plot,
    x="X2",
    y="X1",
    color="Count",
    text="Count",
    facet_col="Design",
    category_orders={"Design": [DISPLAY_NAMES[d] for d in DESIGNS]},
    color_continuous_scale="Cividis",
    range_color=[0, 4],
    labels={"Count": "Count"},
)
fig14a.update_traces(marker={"symbol": "square", "size": 18, "line": {"color": "white", "width": 0.7}}, textposition="middle center")
fig14a.update_layout(width=1120, height=360, template="plotly_white", margin={"l": 75, "r": 30, "t": 60, "b": 60})
fig14a.update_xaxes(title="X2", range=[-3.1, 3.1], dtick=1, gridcolor="rgba(0,0,0,0.08)")
fig14a.update_yaxes(title="X1", range=[-2.1, 2.1], dtick=1, gridcolor="rgba(0,0,0,0.08)")
fig14a.update_coloraxes(colorbar_title="Count")
fig14a.for_each_annotation(lambda a: a.update(text=a.text.replace("Design=", "")))
fig14a.add_annotation(text="a)", xref="paper", yref="paper", x=-0.055, y=1.12, showarrow=False, font={"size": 18, "color": "#111111"})

figure_14a_png_path = OUTPUT_DIR / "figure_14a_design_counts_pydoe.png"
fig14a.write_image(figure_14a_png_path, scale=2)
display(Image(filename=figure_14a_png_path))

# Panel (b): low-interpolation filled contours from sampled Y values.
# DSD and RSM are complete 3x3 X2-X1 grids, so use that grid directly.
# This keeps all corners, edge centers, and the center point visibly anchored.
response_samples = (
    doe.loc[doe["Design"].isin(DESIGNS), ["Design", "X1", "X2", "Y"]]
    .groupby(["Design", "X1", "X2"], as_index=False)["Y"].mean()
)

levels = np.linspace(-1e-6, 1, 11)
fig_static_14b, axes_14b = plt.subplots(1, 3, figsize=(12.2, 3.5), sharex=True, sharey=True, constrained_layout=True)
last_contour = None

for ax, design in zip(axes_14b, DESIGNS):
    data = response_samples.loc[response_samples["Design"] == design].copy()
    x = data["X2"].to_numpy()
    y = data["X1"].to_numpy()
    z = data["Y"].to_numpy()
    unique_x = np.sort(data["X2"].unique())
    unique_y = np.sort(data["X1"].unique())
    is_complete_rectangular_grid = len(unique_x) * len(unique_y) == len(data)

    if is_complete_rectangular_grid:
        # Minimal linear interpolation from the observed 3x3 table.
        z_table = (
            data.pivot(index="X1", columns="X2", values="Y")
            .reindex(index=unique_y, columns=unique_x)
            .to_numpy()
        )
        interpolator = RegularGridInterpolator((unique_y, unique_x), z_table, method="linear")
        dense_x = np.linspace(unique_x.min(), unique_x.max(), 31)
        dense_y = np.linspace(unique_y.min(), unique_y.max(), 25)
        X_dense, Y_dense = np.meshgrid(dense_x, dense_y)
        Z_dense = interpolator(np.column_stack([Y_dense.ravel(), X_dense.ravel()])).reshape(Y_dense.shape)
        last_contour = ax.contourf(X_dense, Y_dense, Z_dense, levels=levels, cmap="viridis", vmin=0, vmax=1, extend="both")
    else:
        # For the space-filling design, keep the triangulated surface between irregular sampled points.
        triangulation = mtri.Triangulation(x, y)
        last_contour = ax.tricontourf(triangulation, z, levels=levels, cmap="viridis", vmin=0, vmax=1, extend="both")

    ax.scatter(x, y, s=18, facecolors="white", edgecolors="#333333", linewidths=0.5, zorder=3)
    ax.set_title(DISPLAY_NAMES[design], fontsize=11)
    ax.set_xlim(-3.1, 3.1)
    ax.set_ylim(-2.1, 2.1)
    ax.set_xlabel("X2")
    ax.grid(alpha=0.18, linewidth=0.8)
    if ax is axes_14b[0]:
        ax.set_ylabel("X1")
    else:
        ax.set_ylabel("")

fig_static_14b.colorbar(last_contour, ax=axes_14b, shrink=0.86, pad=0.015, label="Y (Rosenbrock)")
fig_static_14b.text(0.01, 0.96, "b)", fontsize=14, weight="bold", va="top")
figure_14b_png_path = OUTPUT_DIR / "figure_14b_sampled_response_pydoe.png"
fig_static_14b.savefig(figure_14b_png_path, dpi=220, bbox_inches="tight")
plt.close(fig_static_14b)
display(Image(filename=figure_14b_png_path))

# Hoverable Plotly Express layer: sampled points colored by their measured response.
fig14b = px.scatter(
    response_samples.assign(Design=response_samples["Design"].map(DISPLAY_NAMES)),
    x="X2",
    y="X1",
    color="Y",
    facet_col="Design",
    category_orders={"Design": [DISPLAY_NAMES[d] for d in DESIGNS]},
    color_continuous_scale=PAPER_COLORSCALE,
    range_color=[0, 1],
    labels={"Y": "Y (Rosenbrock)"},
    hover_data={"X1": ":.3f", "X2": ":.3f", "Y": ":.3f"},
)
fig14b.update_traces(marker={"size": 13, "symbol": "square", "line": {"color": "white", "width": 0.7}})
fig14b.update_layout(width=1120, height=380, template="plotly_white", margin={"l": 75, "r": 30, "t": 60, "b": 60})
fig14b.update_xaxes(title="X2", range=[-3.1, 3.1], dtick=1, gridcolor="rgba(0,0,0,0.08)")
fig14b.update_yaxes(title="X1", range=[-2.1, 2.1], dtick=1, gridcolor="rgba(0,0,0,0.08)")
fig14b.update_coloraxes(colorbar_title="Y (Rosenbrock)")
fig14b.for_each_annotation(lambda a: a.update(text=a.text.replace("Design=", "")))
fig14b.add_annotation(text="b) sampled values", xref="paper", yref="paper", x=-0.055, y=1.12, showarrow=False, font={"size": 18, "color": "#111111"})
fig14b_html = fig14b.to_html(include_plotlyjs=False, full_html=False)






**Figure 14a-b footnote.** Panel (a) shows projected design layouts and replicate counts. Panel (b) uses low-interpolation filled contours from the sampled `Y (Rosenbrock)` values. For DSD and RSM, the complete 3x3 sampled response table is interpolated directly so all corners, edge centers, and the center point remain visible; the space-filling panel uses triangulation between its irregular sampled points.

## Figure 14c: LightGBM bootstrap forest with a regenerated SNF

For panel (c), each design is refit 1,000 times. Every repetition regenerates `Random Normal`, appends it to `X1`, `X2`, and inert `X3`, fits a LightGBM random forest (`boosting_type="rf"`), and converts split gain to a contribution fraction. The random feature is the synthetic-noise floor: real factors that repeatedly fall below it should be treated cautiously. The Plotly violin plots use the same factor order as the paper: `X2`, `X1`, `X3`, then `Random Normal`.

In [ ]:
contribution_path = OUTPUT_DIR / "figure_14c_lightgbm_bootstrap_forest_contributions.csv"

if contribution_path.exists():
    # Reuse the validated 1,000-simulation table when rerunning the notebook.
    contributions = pd.read_csv(contribution_path)
    print(f"Loaded cached contribution table from: {contribution_path}")
else:
    contributions = run_bootstrap_forests(n_simulations=1000)
    contributions.to_csv(contribution_path, index=False)
    print(f"Saved contribution table to: {contribution_path}")

contributions["Design"] = pd.Categorical(contributions["Design"], categories=DESIGNS, ordered=True)
contributions["Factor"] = pd.Categorical(contributions["Factor"], categories=FACTOR_NAMES, ordered=True)
display(contributions.head())

In [ ]:
summary = (
    contributions
    .groupby(["Design", "Factor"], observed=True)["Portion"]
    .agg(["mean", "median", "std"])
    .reset_index()
)
summary["mean"] = summary["mean"].round(3)
summary["median"] = summary["median"].round(3)
summary["std"] = summary["std"].round(3)
display(summary)

In [ ]:
# Plotly Express makes the distribution shape explicit with violins.
plot_data = contributions.copy()
plot_data["Factor"] = plot_data["Factor"].astype(str)
plot_data["Design"] = plot_data["Design"].astype(str)
plot_data["color_group"] = np.where(
    plot_data["Factor"].isin(["X3 (non-informative)", "Random Normal"]),
    "Noise / inert",
    "Active factors",
)

fig14c = px.violin(
    plot_data,
    x="Portion",
    y="Factor",
    color="color_group",
    facet_col="Design",
    category_orders={"Factor": FACTOR_NAMES[::-1], "Design": DESIGNS},
    color_discrete_map={"Active factors": "#111111", "Noise / inert": "#b2182b"},
    points=False,
    orientation="h",
    box=False,
    hover_data={"n_simulation": True, "N": True, "color_group": False},
)

fig14c.update_traces(
    side="positive",
    width=0.85,
    meanline_visible=False,
    line={"width": 0.8},
    spanmode="hard",
)
fig14c.update_layout(
    width=1120,
    height=390,
    template="plotly_white",
    showlegend=False,
    margin={"l": 90, "r": 20, "t": 55, "b": 55},
)
fig14c.update_xaxes(range=[0, 1], title="Portion (1,000 bootstrap forest models)", gridcolor="rgba(0,0,0,0.10)")
fig14c.update_yaxes(title="Factor", categoryorder="array", categoryarray=FACTOR_NAMES[::-1])
fig14c.for_each_annotation(lambda a: a.update(text=a.text.replace("Design=", "")))
fig14c.add_annotation(text="c)", xref="paper", yref="paper", x=-0.075, y=1.08, showarrow=False, font={"size": 18, "color": "#111111"})

figure_14c_png_path = OUTPUT_DIR / "figure_14c_lightgbm_bootstrap_forest_pydoe.png"
fig14c.write_image(figure_14c_png_path, scale=2)
display(Image(filename=figure_14c_png_path))
# Figure 14c interactivity is included in the combined Figure 14 notebook output below.

**Figure 14c footnote.** The red variables are the deliberately inert factor `X3` and the regenerated `Random Normal` SNF. The space-filling design most consistently ranks `X1` and `X2` above the random noise floor, while the inert factor stays near the SNF cloud. The DSD and I-optimal RSM runs are more fragile because a small, structured design can let random variation occasionally outrank a real or inert factor.

## Full Figure 14 in one notebook output

The cell below embeds the static paper-style Figure 14 and then displays an inline interactive Plotly version for hover inspection. No separate figure-level HTML files are written; the exported notebook HTML is the single HTML deliverable.

In [ ]:
# Static paper-style Figure 14 embedded in the notebook.
figure_14_static_path = OUTPUT_DIR / "figure_14_static_pydoe_lightgbm.png"
static_panel_paths = [figure_14a_png_path, figure_14b_png_path, figure_14c_png_path]
static_panels = [PILImage.open(path).convert("RGB") for path in static_panel_paths]
max_width = max(img.width for img in static_panels)
total_height = sum(img.height for img in static_panels)
combined_static = PILImage.new("RGB", (max_width, total_height), "white")
y_offset = 0
for img in static_panels:
    combined_static.paste(img, ((max_width - img.width) // 2, y_offset))
    y_offset += img.height
combined_static.save(figure_14_static_path)
display(Image(filename=figure_14_static_path))

# Interactive hover version of the same figure panels.
figure_14_html = """
<div style='font-family: Arial, sans-serif; max-width: 1160px;'>
  <div>{panel_a}</div>
  <div style='margin-top: 10px;'>{panel_b}</div>
  <div style='margin-top: 10px;'>{panel_c}</div>
</div>
""".format(
    panel_a=fig14a.to_html(include_plotlyjs="cdn", full_html=False),
    panel_b=fig14b_html,
    panel_c=fig14c.to_html(include_plotlyjs=False, full_html=False),
)

# Remove intermediate panel image files; the notebook output keeps the embedded static images.
# The exported notebook HTML is the single HTML deliverable.
for intermediate_path in static_panel_paths:
    try:
        intermediate_path.unlink()
    except FileNotFoundError:
        pass
for panel_path in globals().get("figure_14b_panel_paths", []):
    try:
        panel_path.unlink()
    except FileNotFoundError:
        pass

print(f"Saved static Figure 14 PNG to: {figure_14_static_path}")
print("Interactive Figure 14 is embedded inline in the notebook and exported notebook HTML.")
display(HTML(figure_14_html))

## Paper discussion reproduced in plain language

The DoE benchmark highlights a practical limitation of small classical designs. DSD and RSM layouts are efficient when the surface is close to quadratic, but a strongly folded response can hide between their corner, center, and axial points. A single space-filling design is not a final optimizer, but it gives a stronger first look at nonlinear global structure and makes inert factors easier to identify with the SNF screen.

The paper's recommendation is therefore pragmatic: use an initial design to detect curvature, flag inert factors, and seed a surrogate model, then switch to targeted sequential experiments once the likely optimum valley is visible. The 1,000 regenerated-SNF forests are included because tiny DoE tables are statistically fragile; the cloud shows how often random noise can appear competitive with real factors.